# Theo dõi Quá trình Huấn luyện

Notebook này hướng dẫn cách tải và phân tích log huấn luyện từ WandB,
giúp bạn đánh giá quá trình huấn luyện và phát hiện vấn đề sớm.

## Yêu cầu
- Đã đăng nhập WandB
- Đang hoặc đã huấn luyện một run

In [ ]:
import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

MJLAB_DIR = "/home/nguyenld12/Documents/Humanoid_Tracking_Task/mjlab"
os.chdir(MJLAB_DIR)

print("Thư viện đã tải thành công.")

## 1. Kết nối WandB

Tải log huấn luyện từ WandB. Bạn cần biết đường dẫn run: `entity/project/run_id`.

In [ ]:
import wandb

# === SỬA THÔNG TIN RUN TẠI ĐÂY ===
WANDB_ENTITY = "your-entity"     # Tên tổ chức hoặc username
WANDB_PROJECT = "mjlab"          # Tên project
WANDB_RUN_ID = "your-run-id"     # ID run cần phân tích

# Kết nối
api = wandb.Api()
run_path = f"{WANDB_ENTITY}/{WANDB_PROJECT}/{WANDB_RUN_ID}"

try:
    run = api.run(run_path)
    print(f"Tên run: {run.name}")
    print(f"Trạng thái: {run.state}")
    print(f"Tạo lúc: {run.created_at}")
except wandb.errors.CommError:
    print(f"[LỖI] Không tìm thấy run: {run_path}")
    print("Kiểm tra lại WANDB_ENTITY, WANDB_PROJECT, WANDB_RUN_ID.")

In [ ]:
# Tải log
history = run.scan_history()
data = {}
for row in history:
    for key, value in row.items():
        if key not in data:
            data[key] = []
        data[key].append(value)

print(f"Số iteration đã ghi: {len(data.get('_step', []))}")
print(f"\nCác metric có sẵn:")
for key in sorted(data.keys()):
    if not key.startswith('_'):
        print(f"  - {key}")

## 2. Phân tích từ Tensorboard (offline)

Nếu không dùng WandB hoặc muốn phân tích offline, bạn có thể đọc log từ
thư mục local. mjlab ghi log Tensorboard vào thư mục log.

In [ ]:
# Đọc log từ Tensorboard (phương pháp offline)
from glob import glob

# Liệt kê các run đã huấn luyện
log_base = os.path.join(MJLAB_DIR, "logs", "rsl_rl", "m2v6_tracking")
if os.path.exists(log_base):
    runs = sorted(os.listdir(log_base))
    print("Các run có sẵn:")
    for i, r in enumerate(runs):
        print(f"  [{i}] {r}")
else:
    print(f"Chưa có thư mục log: {log_base}")
    print("Hãy huấn luyện ít nhất 1 run trước.")

## 3. Biểu đồ Reward

Reward tổng là chỉ số quan trọng nhất. Nó nên tăng dần theo thời gian.

In [ ]:
# Hàm tiện ích: vẽ biểu đồ
def plot_metric(data, key, title, ylabel, smooth=50):
    """Vẽ biểu đồ metric với đường làm mượt."""
    if key not in data:
        print(f"Không tìm thấy metric: {key}")
        return
    
    values = np.array([v for v in data[key] if v is not None])
    steps = np.arange(len(values))
    
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(steps, values, alpha=0.3, color='blue', label='Thực tế')
    
    # Đường trung bình trượt
    if len(values) > smooth:
        smoothed = np.convolve(values, np.ones(smooth)/smooth, mode='valid')
        ax.plot(steps[smooth-1:], smoothed, color='red', linewidth=2, label=f'Trung bình ({smooth} iteration)')
    
    ax.set_title(title, fontsize=14)
    ax.set_xlabel('Iteration')
    ax.set_ylabel(ylabel)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Thống kê
    print(f"  Min: {values.min():.4f}")
    print(f"  Max: {values.max():.4f}")
    print(f"  Cuối: {values[-1]:.4f}")
    if len(values) >= smooth:
        print(f"  TB {smooth} cuối: {values[-smooth:].mean():.4f}")

In [ ]:
# Vẽ reward tổng
# Tên metric có thể khác tùy cấu hình WandB
# Thử các tên phổ biến:
reward_keys = ['Train/mean_reward', 'mean_reward', 'reward']
for key in reward_keys:
    if key in data:
        plot_metric(data, key, 'Reward trung bình theo thời gian', 'Reward')
        break
else:
    print("Không tìm thấy metric reward. Các key có sẵn:")
    print([k for k in data.keys() if 'reward' in k.lower()])

In [ ]:
# Vẽ episode length
length_keys = ['Train/mean_episode_length', 'mean_episode_length', 'episode_length']
for key in length_keys:
    if key in data:
        plot_metric(data, key, 'Thời lượng Episode trung bình', 'Số bước')
        break
else:
    print("Không tìm thấy metric episode length.")

## 4. Phân tích từng Thành phần Reward

Xem chi tiết từng thành phần reward để hiểu robot đang học tốt/kém ở phần nào.

In [ ]:
# Tìm và vẽ từng thành phần reward
reward_component_keys = [k for k in data.keys() if 'reward' in k.lower() and k != 'mean_reward']

if reward_component_keys:
    n_components = len(reward_component_keys)
    cols = 2
    rows = (n_components + 1) // 2
    
    fig, axes = plt.subplots(rows, cols, figsize=(14, 4 * rows))
    axes = axes.flatten() if n_components > 2 else [axes] if n_components == 1 else axes
    
    for i, key in enumerate(sorted(reward_component_keys)):
        values = np.array([v for v in data[key] if v is not None])
        if len(values) > 0:
            ax = axes[i]
            ax.plot(values, alpha=0.5)
            if len(values) > 50:
                smoothed = np.convolve(values, np.ones(50)/50, mode='valid')
                ax.plot(range(49, len(values)), smoothed, color='red', linewidth=2)
            short_name = key.split('/')[-1]
            ax.set_title(short_name)
            ax.grid(True, alpha=0.3)
    
    # Ẩn axes thừa
    for i in range(len(reward_component_keys), len(axes)):
        axes[i].set_visible(False)
    
    plt.suptitle('Từng thành phần Reward', fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("Không tìm thấy các thành phần reward riêng lẻ.")
    print("Có thể WandB chưa ghi chi tiết. Kiểm tra cấu hình logger.")

## 5. Phát hiện Vấn đề

Dưới đây là các dấu hiệu và cách xử lý vấn đề phổ biến.

In [ ]:
# Phân tích tự động
def diagnose_training(data):
    """Phân tích tự động và đưa ra khuyến nghị."""
    print("=" * 50)
    print("  CHẨN ĐOÁN HUẤN LUYỆN")
    print("=" * 50)
    
    # Tìm metric reward
    reward_key = None
    for k in ['Train/mean_reward', 'mean_reward', 'reward']:
        if k in data:
            reward_key = k
            break
    
    if reward_key is None:
        print("Không tìm thấy metric reward.")
        return
    
    rewards = np.array([v for v in data[reward_key] if v is not None])
    n = len(rewards)
    
    if n < 100:
        print(f"\nChỉ có {n} iteration — quá ít để chẩn đoán. Hãy huấn luyện thêm.")
        return
    
    # 1. Xu hướng tổng thể
    first_quarter = rewards[:n//4].mean()
    last_quarter = rewards[-n//4:].mean()
    improvement = last_quarter - first_quarter
    
    print(f"\n1. Xu hướng tổng thể:")
    print(f"   Reward TB 1/4 đầu:  {first_quarter:.4f}")
    print(f"   Reward TB 1/4 cuối: {last_quarter:.4f}")
    print(f"   Cải thiện: {improvement:+.4f}")
    
    if improvement < 0:
        print("   ⚠ CẢNH BÁO: Reward giảm! Có thể divergence.")
        print("   → Thử giảm learning rate hoặc kiểm tra motion data.")
    elif improvement < 0.5:
        print("   ⚠ Cải thiện chậm.")
        print("   → Thử tăng num-envs hoặc huấn luyện lâu hơn.")
    else:
        print("   ✓ Xu hướng tốt!")
    
    # 2. Plateau
    if n > 500:
        recent = rewards[-200:]
        recent_std = recent.std()
        recent_trend = np.polyfit(range(len(recent)), recent, 1)[0]
        
        print(f"\n2. Kiểm tra Plateau (200 iteration cuối):")
        print(f"   Xu hướng: {recent_trend:+.6f}/iteration")
        print(f"   Dao động: {recent_std:.4f}")
        
        if abs(recent_trend) < 0.001 and recent_std < 0.5:
            print("   ⚠ Có vẻ đã plateau — reward không tăng nữa.")
            print("   → Nếu reward đủ cao: dừng huấn luyện.")
            print("   → Nếu chưa đủ: thử điều chỉnh reward weights.")
        else:
            print("   ✓ Vẫn đang cải thiện.")
    
    # 3. Variance
    print(f"\n3. Độ ổn định:")
    overall_std = rewards.std()
    print(f"   Độ lệch chuẩn tổng: {overall_std:.4f}")
    if overall_std > 2.0:
        print("   ⚠ Dao động rất lớn — huấn luyện không ổn định.")
        print("   → Tăng mini-batches hoặc giảm learning rate.")
    else:
        print("   ✓ Ổn định.")

# Chạy chẩn đoán
diagnose_training(data)

## 6. Khi nào nên Dừng Huấn luyện?

Không có câu trả lời chính xác. Nhưng có một số gợi ý:

| Điều kiện | Hành động |
|-----------|----------|
| Reward đã hội tụ (không tăng trong 2000+ iteration) | Dừng, dùng checkpoint tốt nhất |
| Episode length gần max (10s = 500 bước) | Dừng — robot đã sống sót tốt |
| Reward bắt đầu giảm | Dừng, lùi về checkpoint trước |
| Đã chạy 30,000 iteration | Dừng, đánh giá kết quả |

## Bước tiếp theo

Sau khi phân tích, chuyển sang đánh giá policy:
- [doc 08 - Đánh giá Policy](../docs/08-danh-gia-policy.md)
- Hoặc chạy script: `bash scripts/03_chay_policy.sh`